# Práctica 3 · Segmentación: ¿qué tipos de incidentes hay?
**Analítica de procesos para ciberseguridad**

**El problema.** Tenemos miles de incidentes y nadie los va a mirar uno por uno. ¿Podemos **agruparlos en pocos perfiles** según **cómo se atienden** (cuánto tardan, cuántas veces cambian de manos, cuánto esperan), para tratar distinto a cada perfil?

**La tarea.** A diferencia de la Práctica 1, aquí **no hay respuesta correcta**: nadie nos dijo «este incidente es del tipo A». El modelo tiene que **descubrir los grupos** por su cuenta. Se llama **aprendizaje no supervisado**, y esta tarea, **segmentación** (o *clustering*).

**Una analogía.** Imagina que organizas una fiesta y tienes que sentar a 200 personas en 5 mesas **sin saber nada de ellas**, solo mirando cómo se comportan: quién habla con quién, quién baila, quién se queda aparte. Intentas que **las personas de una mesa se parezcan entre sí y se diferencien de las de otras mesas**. Eso hace un algoritmo de segmentación.

**Por qué importa en seguridad.** Los perfiles sirven para asignar equipos, fijar plazos distintos, auditar los casos raros o saber qué «normal» comparar (la Práctica 4 usa esa idea).

**Los datos.** El mismo registro de incidentes de la Práctica 2 (mesa de servicio de TI, ServiceNow; 24.916 incidentes). Allí vimos el **proceso**; aquí lo resumimos en **una fila por incidente**.

> **Fuente y cambios.** Amaral, Fantinato, Reijers y Peres (2018). *Incident management process enriched event log.* UCI Machine Learning Repository, https://doi.org/10.24432/C57S4H. Licencia CC BY 4.0. Cambios hechos con fines académicos (traducción al español, formato de fechas, 2 incidentes inválidos fuera): ver `scripts/preparar_servicenow.py`.

## 0 · Preparación

In [ ]:
# Importa las librerías: pandas (tablas), numpy (cálculos) y matplotlib (gráficos)
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")                # oculta advertencias para que la salida se lea limpia
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# Dirección de los datos: repositorio de GitHub y versión fija de donde se descargan los archivos
REPO = "lsandinop/analitica-procesos-cybersecurity"
VERSION = "main"
DATA_BASE = "" if REPO.startswith("USUARIO") else f"https://raw.githubusercontent.com/{REPO}/{VERSION}/datos/"

def cargar(nombre, fechas=None):
    # Lee un archivo CSV y lo devuelve como tabla (DataFrame). Lo busca primero en GitHub;
    # si no está disponible, en la carpeta local. `fechas` son las columnas que se leen como fecha y hora.
    if DATA_BASE:
        try:
            return pd.read_csv(DATA_BASE + nombre, parse_dates=fechas)
        except Exception:
            print("No pude descargar el archivo de GitHub; busco una copia local.")
    for ruta in (nombre, "datos/" + nombre, "../datos/" + nombre, "../../datos/" + nombre):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, parse_dates=fechas)
    raise FileNotFoundError(f"No encuentro {nombre}: súbelo a la carpeta del cuaderno")

eventos = cargar("incidentes_servicenow.csv", fechas=["fecha_evento", "fecha_apertura", "fecha_resolucion", "fecha_cierre"])
print(f"{len(eventos):,} eventos · {eventos.incidente.nunique():,} incidentes".replace(",", "."))

## 1 · De los eventos a una fila por incidente
Para agrupar incidentes necesitamos **una fila por incidente**, con números que describan **cómo se atendió**. Resumimos su historia de eventos en ocho variables:

| Variable | Qué cuenta |
|---|---|
| `cambios_de_estado` | Cuántas veces cambió de estado (Nuevo, En curso, Resuelto...) |
| `reasignaciones` | Cuántas veces cambió de grupo o de analista |
| `reaperturas` | Cuántas veces se reabrió después de resuelto |
| `modificaciones` | Cuántas veces se editó el incidente (una medida de «cuánto trabajo dio») |
| `esperas` | Cuántas veces quedó esperando a alguien (usuario, proveedor, evidencia) |
| `grupos_distintos` | Por cuántos grupos de trabajo pasó |
| `dias_a_resolver` | Días desde que se abrió hasta que se resolvió |
| `prioridad` | 1 (crítica) a 4 (baja) |

`cumplio_sla` (si cumplió el plazo) **no se usa para agrupar**: la dejamos aparte para comprobar, al final, si los grupos sirven.

In [ ]:
def tabla_por_incidente(eventos):
    # Resume el registro de eventos en UNA fila por incidente, con variables que describen CÓMO se atendió
    ev = eventos.sort_values(["incidente", "fecha_evento"], kind="stable")
    g = ev.groupby("incidente")
    cambios = ev[ev.estado != g.estado.shift()]                     # solo las filas donde cambia el estado
    t = pd.DataFrame({
        "cambios_de_estado": cambios.groupby("incidente").size(),
        "reasignaciones": g.reasignaciones.max(),
        "reaperturas": g.reaperturas.max(),
        "modificaciones": g.modificaciones.max(),
        "esperas": cambios.estado.str.startswith("Esperando").groupby(cambios.incidente).sum(),
        "grupos_distintos": g.grupo_asignado.nunique(),
        "dias_a_resolver": (g.fecha_resolucion.last() - g.fecha_apertura.first()).dt.total_seconds() / 86400,
        "prioridad": g.prioridad.last().str[0].astype(float),          # «3 - Moderada» -> 3
        "cumplio_sla": g.cumplio_sla.last()})
    return t.dropna()

tabla = tabla_por_incidente(eventos)
VARIABLES = ["cambios_de_estado", "reasignaciones", "reaperturas", "modificaciones", "esperas", "grupos_distintos", "dias_a_resolver", "prioridad"]
print(f"{len(tabla):,} incidentes con datos completos (de {eventos.incidente.nunique():,}; se pierden los que no tienen fecha de resolución)".replace(",", "."))
tabla.head(8)

In [ ]:
tabla[VARIABLES].describe().round(2).T

**Mira esa tabla.** La mediana de `dias_a_resolver` es menos de 1 día, pero el máximo pasa de 300; `modificaciones` tiene mediana 4 y máximo 129. **Casi todos los incidentes son parecidos y muy pocos son extremos.** Eso afecta a cómo preparamos los datos.

## 2 · Preparar las variables para agrupar
Un algoritmo de segmentación mide **distancias** entre incidentes. Dos problemas:
1. **Las escalas son distintas:** 8 días contra 3 cambios de estado. Si no hacemos nada, la variable con números grandes (días) **domina** la distancia. Se arregla **estandarizando** (pasar cada variable a «cuántas desviaciones se aleja del promedio»).
2. **Las variables están muy sesgadas:** pocos casos extremos estiran la escala. Se arregla con el **logaritmo**, que acerca los valores grandes (`log(1 + x)`).

Así se ve el efecto en los días:

In [ ]:
from sklearn.preprocessing import StandardScaler

logaritmo = np.log1p(tabla[VARIABLES])                              # log(1 + x): acerca los valores grandes, sin problema con los ceros
Z = StandardScaler().fit_transform(logaritmo)                       # cada variable: media 0 y desviación 1

fig, ejes = plt.subplots(1, 3, figsize=(13, 3.3))
ejes[0].hist(tabla.dias_a_resolver, bins=60, color="#1f6db0"); ejes[0].set_title("Días a resolver, tal cual\n(casi todo pegado a cero)", fontsize=10)
ejes[1].hist(logaritmo.dias_a_resolver, bins=60, color="#1f6db0"); ejes[1].set_title("Con logaritmo\n(se ve la forma)", fontsize=10)
ejes[2].hist(Z[:, VARIABLES.index("dias_a_resolver")], bins=60, color="#1f6db0"); ejes[2].set_title("Logaritmo y estandarizado\n(lo que ve el algoritmo)", fontsize=10)
plt.tight_layout(); plt.show()

## 3 · ¿Cuántos grupos?
*K-means*, el algoritmo más usado, necesita que le digamos **cuántos grupos** (k) buscar. Funciona así: coloca k «centros» al azar, asigna cada incidente al centro más cercano, mueve cada centro al promedio de su grupo, y repite hasta que nada cambia (en la fiesta: el centro de cada mesa es «la persona promedio» de esa mesa).

¿Cómo escoger k? Dos medidas habituales:
- **Inercia (el «codo»):** qué tan lejos quedan los incidentes de su centro. Siempre baja al aumentar k; se busca el punto donde **deja de bajar tanto**.
- **Silueta:** de −1 a 1, qué tan bien separado está cada incidente de los grupos vecinos. Más alta es mejor.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rng = np.random.RandomState(0)
muestra = rng.choice(len(Z), 6000, replace=False)                   # la silueta es lenta; se calcula con una muestra de 6.000 incidentes
valores_k = range(2, 9)
inercia, silueta = [], []
for k in valores_k:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    inercia.append(km.inertia_); silueta.append(silhouette_score(Z[muestra], km.labels_[muestra]))

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
ejes[0].plot(list(valores_k), inercia, "o-", color="#1f6db0"); ejes[0].set_title("Inercia (busca el «codo»)"); ejes[0].set_xlabel("número de grupos (k)")
ejes[1].plot(list(valores_k), silueta, "o-", color="#d9534f"); ejes[1].set_title("Silueta (más alta = mejor)"); ejes[1].set_xlabel("número de grupos (k)"); ejes[1].set_ylim(0.3, 0.5)
plt.tight_layout(); plt.show()
print("Silueta por k:", {k: round(s, 3) for k, s in zip(valores_k, silueta)})

**Qué leer ahí.** **No hay un k claramente mejor**: el codo es suave y la silueta casi no cambia entre 2 y 8 grupos (todas rondan 0,4). Es lo normal: **los datos reales no vienen en grupos nítidos**. Cuando la métrica no decide, decide la **utilidad**: ¿qué número de grupos produce perfiles que podamos **describir y usar**? Comparemos 4 y 5.

In [ ]:
def perfiles(k):
    # Agrupa con k-means y resume cada grupo (medianas de las variables). Los grupos se numeran de menor a mayor tiempo de resolución.
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    t = tabla.assign(grupo=km.labels_)
    orden = t.groupby("grupo").dias_a_resolver.median().sort_values().index
    t["grupo"] = t.grupo.map({g: i for i, g in enumerate(orden)})
    resumen = t.groupby("grupo").agg(incidentes=("prioridad", "size"), cambios=("cambios_de_estado", "median"), reasignaciones=("reasignaciones", "median"),
                                      modificaciones=("modificaciones", "median"), esperas=("esperas", "mean"), dias=("dias_a_resolver", "median"),
                                      reaperturas=("reaperturas", "mean"), prioridad=("prioridad", "mean"), pct_incumple_sla=("cumplio_sla", lambda s: (1 - s.mean()) * 100))
    return t, resumen.round(2)

_, perfil4 = perfiles(4)
print("Con 4 grupos:"); perfil4

In [ ]:
tabla_k5, perfil5 = perfiles(5)
print("Con 5 grupos:"); perfil5

**Qué comparar.** Con 4 grupos aparecen: incidentes rápidos, los que cambian de manos, los que **esperan** y los **reabiertos**. Con 5, aparece **uno más**: un grupo pequeño de **prioridad alta** (prioridad promedio ≈ 1,6, frente a ≈ 3 en los demás) que **casi siempre incumple el plazo** (99 %). Ese grupo se escondía dentro de otro, y es **útil** (nos dice que el plazo de los casos urgentes es el que más se rompe). Nos quedamos con **5**.

## 4 · Ver los grupos
Tenemos 8 variables por incidente, y no podemos dibujar 8 dimensiones. El **análisis de componentes principales (PCA)** resume las 8 variables en 2 «ejes» que conservan la mayor parte de la variación, y así se puede hacer un mapa plano. Cada punto es un incidente, coloreado por su grupo:

In [ ]:
from sklearn.decomposition import PCA

NOMBRES = {0: "Rutina rápida", 1: "Prioridad alta", 2: "Cambian de manos, sin esperas", 3: "Con esperas", 4: "Reabiertos y complejos"}     # nombres que damos al leer los perfiles
pca = PCA(n_components=2, random_state=0).fit(Z)
plano = pca.transform(Z)
print(f"Los 2 ejes conservan el {pca.explained_variance_ratio_.sum():.0%} de la variación de las 8 variables")

colores = ["#5cb85c", "#d9534f", "#5b9bd5", "#f0ad4e", "#8e44ad"]
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ver = rng.choice(len(Z), 5000, replace=False)                        # se dibujan 5.000 puntos al azar para no saturar el gráfico
for g in range(5):
    sel = ver[tabla_k5.grupo.values[ver] == g]
    ax.scatter(plano[sel, 0], plano[sel, 1], s=7, alpha=0.5, color=colores[g], label=NOMBRES[g])
ax.legend(markerscale=2.5, fontsize=8); ax.set_xlabel("eje 1"); ax.set_ylabel("eje 2"); ax.set_title("Los 5 grupos de incidentes (cada punto es un incidente)")
plt.tight_layout(); plt.show()

**Qué leer ahí.** El mapa no separa limpiamente los grupos, y el de «prioridad alta» casi se pierde entre los demás. Es normal: los dos ejes resumen sobre todo **los tiempos y los traspasos**, que es donde hay más variación, y la prioridad casi no varía. **Un mapa en 2 dimensiones es un resumen**; los grupos se distinguen mejor en la **tabla de perfiles** de abajo.

## 5 · Describir cada grupo
Un grupo sin nombre no sirve para decidir. Leyendo la tabla de perfiles, le dimos un nombre a cada uno (puedes cambiarlos en el diccionario `NOMBRES` de arriba). ¿Qué tan grande es cada uno y qué tan mal le va con el plazo?

In [ ]:
resumen = perfil5.assign(nombre=[NOMBRES[g] for g in perfil5.index])[["nombre", "incidentes", "dias", "reasignaciones", "esperas", "prioridad", "pct_incumple_sla"]]
resumen["% de los incidentes"] = (resumen.incidentes / resumen.incidentes.sum() * 100).round(1)
print(resumen.to_string())

fig, ejes = plt.subplots(1, 2, figsize=(12, 3.6))
ejes[0].barh([NOMBRES[g] for g in perfil5.index][::-1], perfil5.incidentes[::-1], color=colores[::-1]); ejes[0].set_title("Cuántos incidentes hay en cada grupo", fontsize=10)
ejes[1].barh([NOMBRES[g] for g in perfil5.index][::-1], perfil5.pct_incumple_sla[::-1], color=colores[::-1]); ejes[1].set_xlim(0, 105)
ejes[1].set_title("% que incumple el plazo (SLA)", fontsize=10); ejes[1].set_yticklabels([])
plt.tight_layout(); plt.show()

**Qué leer ahí.** Los grupos **se parecen por dentro y se diferencian por fuera**, y **no usamos el SLA para formarlos**, pero el incumplimiento cambia muchísimo entre ellos (de 13 % a 99 %). Eso es lo que hace **útil** a la segmentación: los grupos capturan algo real del proceso. Y dicen **qué hacer distinto** con cada uno: la rutina rápida casi no necesita atención; los que esperan información piden otra política; los de prioridad alta, un plazo realista o más recursos.

## 6 · ¿Qué tan confiables son los grupos?
No hay respuesta correcta con la cual comparar, pero sí hay dos pruebas de sensatez:
1. **Estabilidad:** si repito el algoritmo con otro sorteo inicial, ¿sale lo mismo? Se mide con el *índice de Rand ajustado* (ARI): 1 = idéntico, 0 = como al azar.
2. **Otro método:** si uso un algoritmo distinto (agrupamiento jerárquico), ¿encuentra grupos parecidos?

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

referencia = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Z).labels_
# 1) Estabilidad: otros 4 sorteos iniciales
aris = [adjusted_rand_score(referencia, KMeans(n_clusters=5, n_init=10, random_state=s).fit(Z).labels_) for s in (1, 2, 3, 4)]
print("Estabilidad (ARI contra otros sorteos):", np.round(aris, 2), "· 1 = idéntico")

# 2) Otro método: agrupamiento jerárquico (muy lento con miles de datos: se usa una muestra de 4.000 incidentes)
sub = rng.choice(len(Z), 4000, replace=False)
jerarquico = AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(Z[sub])
print(f"Acuerdo con el agrupamiento jerárquico (ARI): {adjusted_rand_score(referencia[sub], jerarquico):.2f}")

**Qué leer ahí.** Con otros sorteos, los grupos casi no cambian (ARI ≈ 0,9 o más): son **estables**. Con otro algoritmo el acuerdo es **menor**: otro método agrupa de otra manera. Eso no significa que alguno esté mal: la segmentación **depende del método y de las variables** que se escojan. Por eso lo que importa es si los grupos **sirven para decidir**.

## Cierre
- **Segmentar** = descubrir grupos **sin etiqueta**; no hay respuesta correcta.
- Antes de agrupar hay que **preparar**: escalar y reducir el sesgo (logaritmo).
- **No hay un k «correcto»**: las métricas orientan y la **utilidad** decide.
- Un buen grupo se **describe** (perfil), se **nombra** y **sirve para decidir**.
- Revisar **estabilidad** y comparar con otro método da confianza, pero no certeza.
- En la **Práctica 4** usaremos la idea de «normal para su grupo» para buscar los incidentes que **no encajan en ningún perfil**.